# A coarse space with one basis function

Can correcting a single function improve a preconditioner for the whole finite element system? On the unit square $\Omega=(0,1)^2$ we solve $-\Delta u+\varepsilon u=f$ with homogeneous Neumann boundary conditions and $\varepsilon>0$. For continuous piecewise linear elements, the discrete problem is to find $u_h\in V_h\subset H^1(\Omega)$ such that

$$
\begin{align*}
a_\varepsilon(u_h,v_h)
&=\int_\Omega \bigl(\nabla u_h\cdot\nabla v_h+\varepsilon u_hv_h\bigr)\,dx
=\int_\Omega fv_h\,dx
\qquad\text{for every }v_h\in V_h.
\end{align*}
$$

At $\varepsilon=0$, the constants form the Neumann kernel. For $\varepsilon>0$ the matrix is positive definite, but the constant function has energy squared $a_\varepsilon(1,1)=\varepsilon|\Omega|$. We use $f(x,y)=1+x^2y$. Testing with $v_h=1$ gives $\varepsilon\int_\Omega u_h\,dx=\int_\Omega f\,dx$. The load thus excites the constant mode as well as spatial variation. Its nonzero integral also explains why the pure Neumann problem at $\varepsilon=0$ would have no solution for this load.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ngsolve import *
from ngsolve.meshes import MakeStructured2DMesh

n = 10
epsilon_values = [1.0, 0.1, 0.01, 0.001, 1e-4, 1e-5]
tol = 1e-8

mesh = MakeStructured2DMesh(quads=False, nx=n, ny=n)
fes = H1(mesh, order=1)
u, v = fes.TnT()
load = LinearForm((1 + x**2 * y) * v * dx(bonus_intorder=2))
load.Assemble()

constant = GridFunction(fes)
constant.Set(1)
e = constant.vec
print(f"{fes.ndof} free coefficients on the Neumann mesh")

## An exact solve on the constants

Let $A$ be the matrix of $a_\varepsilon$, let $D=\operatorname{diag}(A)$, and let $e$ represent the constant function $1$. For a residual $r$, we seek a coarse correction $\alpha e$. Testing the corrected residual against $e$ gives

$$
\begin{align*}
e^\top(r-\alpha Ae)&=0,
&
\alpha&=\frac{e^\top r}{e^\top A e}.
\end{align*}
$$

The Jacobi correction is $D^{-1}r$. Adding the coarse correction computed from the same residual defines the inverse action $B=C^{-1}$ from Chapter 7,

$$
\begin{align*}
B r=D^{-1}r+e\,\frac{e^\top r}{e^\top A e}.
\end{align*}
$$

The rank-one term solves the coarse problem exactly. The Jacobi term can also contribute along $e$. The denominator is $a_\varepsilon(1,1)=\varepsilon|\Omega|$. In NGSolve, `BaseMatrix(e)` represents the map from a scalar $\alpha$ to the vector $\alpha e$. Its transpose takes the scalar product with $e$.

In [ ]:
def inverse_actions(A):
    jacobi = A.CreateSmoother(fes.FreeDofs())
    E = BaseMatrix(e)
    coarse_energy = InnerProduct(e, A * e)
    coarse = (1.0 / coarse_energy) * (E @ E.T)
    return {"Jacobi": jacobi, "Jacobi + constant": jacobi + coarse}

## Compare on the same right-hand side

We start preconditioned conjugate gradients (PCG) at $U^{(0)}=0$ with either inverse action. For both runs we stop as soon as the explicitly recomputed residual satisfies $\|b-AU^{(k)}\|_2/\|b\|_2\le 10^{-8}$. The residual in the PCG recurrence is updated separately, as in Chapter 7.

In [ ]:
def pcg(A, b, C_inverse, tol, max_steps=500):
    U = b.CreateVector()
    U[:] = 0
    r = b.Copy()
    initial_norm = Norm(r)
    if initial_norm == 0:
        return U, [0.0]

    z = b.CreateVector()
    z.data = C_inverse * r
    d = z.Copy()
    rho = InnerProduct(r, z)
    residuals = [1.0]
    q = b.CreateVector()

    for k in range(max_steps):
        q.data = A * d
        tau = rho / InnerProduct(d, q)
        U.data += tau * d
        r.data -= tau * q

        residuals.append(Norm(b - A * U) / initial_norm)
        if residuals[-1] <= tol:
            return U, residuals

        z.data = C_inverse * r
        rho_new = InnerProduct(r, z)
        d = (z + (rho_new / rho) * d).Evaluate()
        rho = rho_new

    raise RuntimeError("PCG reached its iteration limit")

## Which eigenvalues change?

Only the reaction coefficient varies in the next experiment. We keep the mesh small enough to compute every eigenvalue. If $A=LL^\top$ is its Cholesky factorization, the preconditioned matrix $BA$ is similar to the symmetric matrix $L^\top B L$. The ratio of its largest and smallest eigenvalues is the condition number in the PCG bound. Predict which end of the Jacobi spectrum will change most as $\varepsilon$ decreases.

In [ ]:
histories = {}
spectra = {}
print(f"{'epsilon':>8}   {'inverse action':<18} {'cond. no.':>12} {'PCG steps':>10}")
for epsilon in epsilon_values:
    a = BilinearForm((grad(u) * grad(v) + epsilon * u * v) * dx)
    a.Assemble()
    A = a.mat
    L = np.linalg.cholesky(np.array(A.ToDense()))

    for name, B in inverse_actions(A).items():
        eigenvalues = np.linalg.eigvalsh(L.T @ np.array(B.ToDense()) @ L)
        _, history = pcg(A, load.vec, B, tol)
        spectra[epsilon, name] = eigenvalues
        histories[epsilon, name] = history
        kappa = eigenvalues[-1] / eigenvalues[0]
        print(f"{epsilon:8.3g}   {name:<18} {kappa:12.2e} {len(history)-1:10d}")

In [ ]:
epsilon = min(epsilon_values)
fig, (ax_spec, ax_res) = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True)
for name in ("Jacobi", "Jacobi + constant"):
    low = spectra[epsilon, name][:8]
    ax_spec.semilogy(range(1, len(low) + 1), low, "o-", label=name)
    history = histories[epsilon, name]
    ax_res.semilogy(range(len(history)), history, label=name)

ax_spec.set(xlabel="Eigenvalue index", ylabel="Preconditioned eigenvalue",
            title="Eight smallest eigenvalues", xticks=range(1, 9))
ax_res.axhline(tol, color="0.45", linestyle=":", label="Tolerance")
ax_res.set(xlabel="PCG step", ylabel="True relative residual", title="Convergence")
for ax in (ax_spec, ax_res):
    ax.grid(True, which="major", alpha=0.3)
    ax.legend()
fig.suptitle(rf"Neumann problem, $\varepsilon={epsilon:g}$")
plt.show()

Further reading: [iFEM discussion of a small reaction term](https://jschoeberl.github.io/iFEM/subspacecorrection/examples.html).